[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/thin_lens/thin_lens.ipynb)

# Thin Lenses as Maps on Rays

This example traces bundles of light rays through thin lenses, a wedge prism, and a flat mirror in the plane. An ideal thin lens bends each crossing ray in proportion to its distance from the lens centre, bringing all rays from an object point together at one image point and focusing parallel rays at its focal point. Even when multiple elements are combined, slid along the bench, or tilted off axis, rays from a single object point continue to intersect at a single sharp image.

Each ray is modelled as an antibivector line joined from two points, and an ideal thin lens in the home plane `home` acts linearly on lines as `Line - (Line & origin) * (home / focal)`, shearing each line by its incidence with the lens centre. Conjugating by a motor `motor >> element(motor << Line)` places and tilts any lens, prism, or mirror in the plane, and composing element maps yields a single linear operator on the whole ray bundle. Because lines are global geometric objects, free space between elements needs no propagation map of its own, and the image is simply the meet of any two outgoing rays.

In paraxial ray-transfer matrix notation, a ray is parameterized locally by height and small-angle slope $(y, \theta)$ at a chosen transverse reference plane $x = x_k$, each thin lens $\bigl(\begin{smallmatrix} 1 & 0 \\ -1/f & 1 \end{smallmatrix}\bigr)$ must be interleaved with explicit free-space drift matrices $\bigl(\begin{smallmatrix} 1 & d \\ 0 & 1 \end{smallmatrix}\bigr)$ across every axial gap $d$, and off-axis tilts, wedge deflections, or folded mirror paths require inhomogeneous $3 \times 3$ augmented $(y, \theta, 1)$ matrices and manual optical-axis re-referencing after reflection.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce
from itertools import accumulate

import numpy as np

from examples.optics.thin_lens import render
from examples.util.animation import save_figure, show_states
from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga

mv = NumpyContext(ga).multivector
# Rays are joins of points, and element planes are vectors.
Line = ga.gatype.antibivector()                                                 # Antibivector-
Plane = ga.gatype.vector()                                                      # Vector+
Point = ga.gatype.antivector()                                                  # Antivector-
# Two lines meet in an extrinsic bivector crossing.
Crossing = ga.gatype.bivector()                                                 # Bivector+
LineMap = ga.gatype((Line, Line))                                               # Line- <- Line-
Motor = ga.gatype.rotor()                                                       # Rotor+
origin = mv.point([0, 0])                                                       # [] Point-
# Every optical element is defined in the home plane x == 0.
home = mv.x                                                                     # [] Plane+

## Single-lens imaging and two-lens focusing

Conjugating the home-plane shear by the translator onto a vertical lens plane places a thin lens at any axial station. Rays joined from an object point across a pupil on the first lens refract through one common image crossing `refracted[0] ^ refracted[1]` at the signed distance predicted by the thin-lens equation, while composing two lenses into `second_lens(first_lens)` focuses horizontal rays onto the system's Gullstrand back focal point.

In [ ]:
first_station, first_focal = 1.5, 1.0
second_station, second_focal = 2.1, 0.5
subject = mv.point([-2.0, 0.5])                                                 # [] Point-
pupil_height, n_rays = 0.8, 9

In [ ]:
def thin_lens_map(plane: Plane, focal: float) -> LineMap:
    """Translating the home-plane shear `Line - (Line & origin) * (home / focal)` onto vertical `plane` forms the thin-lens ray map (`[] Line- <- Line-`)."""
    shear = Line - (Line & origin) * (home / focal)                             # [] Line- <- Line-
    shift = (mv.wx * (plane & origin) * 0.5).exp()                              # [] Rotor+
    return shift >> shear(shift << Line)


first_plane, second_plane = mv.x - mv.w * first_station, mv.x - mv.w * second_station  # [] Plane+, Plane+
first_lens = thin_lens_map(first_plane, first_focal)                            # [] Line- <- Line-
second_lens = thin_lens_map(second_plane, second_focal)                         # [] Line- <- Line-
# Join the subject with pupil points on the first lens plane; any two refracted rays meet at the image.
axis = origin & mv.direction([1, 0])                                            # [] Line-
pupil = (first_plane ^ axis) + mv.direction([0, 1]) * np.linspace(-pupil_height, pupil_height, n_rays)  # [n_rays] Point-
rays = subject & pupil                                                          # [n_rays] Line-
refracted = first_lens(rays)                                                    # [n_rays] Line-
image = refracted[0] ^ refracted[1]                                             # [] Crossing+
# Composing both lens maps focuses horizontal rays onto the back focal point.
system = second_lens(first_lens)                                                # [] Line- <- Line-
parallel = mv.direction([1, 0]) & pupil                                         # [n_rays] Line-
between_lenses = first_lens(parallel)                                           # [n_rays] Line-
focused = system(parallel)                                                      # [n_rays] Line-
focus = focused[0] ^ focused[1]                                                 # [] Crossing+
imaging = stack([rays, refracted])                                              # [2, n_rays] Line-
focusing = stack([parallel, between_lenses, focused])                           # [3, n_rays] Line-
save_figure(render.lenses_figure(subject, imaging, image, focusing, focus, stack([first_plane, second_plane])), "thin_lens");

In the upper panel, orange incident rays from the off-axis object refract at the grey lens plane into blue rays converging at the inverted real image; in the lower panel, horizontal orange rays converge partially in green between the two lenses and meet in blue at the combined back focal point.

## Moving optical train of lenses, prism, and mirror

Defining a converging lens, a thin prism (shearing every line by its incidence with an ideal vertical direction so all heights deflect alike), a diverging lens, and a flat mirror (`home.normalized() >> Line`) in the home plane lets an arbitrary motor place and tilt each element. As the first lens slides and tilts and the mirror rocks, the composed train map stays a projective line collineation, so concurrent object rays still meet at a single moving image point.

In [ ]:
train_subject = mv.point([-1.0, 0.5])                                           # [] Point-
aim = mv.point([1.0, 0.0])                                                      # [] Point-
fan_height, n_fan_rays = 0.6, 7
stations = np.array([1.0, 1.9, 2.2, 3.2])
slide_amplitude, lens_tilt_amplitude, rock_amplitude, mirror_tilt = 0.3, 0.3, 0.1, np.pi / 4
n_frames, frame_duration_ms = 108, 40

In [ ]:
train_elements = (
    Line - (Line & origin) * (home / 1.0),                                      # [] Line- <- Line-
    Line - (Line & mv.direction([0, 1])) * home * 0.15,                         # [] Line- <- Line-
    Line - (Line & origin) * (home / -2.0),                                     # [] Line- <- Line-
    home.normalized() >> Line,                                                  # [] Line- <- Line-
)


def optical_train_trace(
    rays: Line, motors: tuple[Motor, ...], elements: tuple[LineMap, ...],
) -> tuple[Plane, Line, LineMap]:
    """Conjugating `elements` (`n_elements` of `[] Line- <- Line-`) by `motors` traces `rays` (`[n_rays] Line-`) into element planes (`[n_elements] Plane+`), per-stage ray bundles (`[n_elements + 1, n_rays] Line-`), and the composed train map (`[] Line- <- Line-`)."""
    placed = [motor >> element(motor << Line) for motor, element in zip(motors, elements)]
    legs = stack(list(accumulate(placed, lambda bundle, element: element(bundle), initial=rays)))  # [..., ...] Line-
    composed = reduce(lambda so_far, element: element(so_far), placed, Line)    # [] Line- <- Line-
    return stack([motor >> home for motor in motors]), legs, composed


def moving_train_states(n_frames: int) -> Iterator[tuple[Point, Plane, Line, LineMap, Crossing]]:
    """Yields `(train_subject, planes, legs, composed, picture)` across `n_frames` phases as the first lens slides and tilts and the mirror rocks."""
    fan = train_subject & (aim + mv.direction([0, 1]) * np.linspace(-fan_height, fan_height, n_fan_rays))  # [n_fan_rays] Line-
    stops = (mv.wx * (-stations / 2)).exp()                                     # [4] Rotor+
    for phase in np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False):
        lens_slide = (mv.wx * (-slide_amplitude * np.sin(phase) / 2)).exp()     # [] Rotor+
        lens_turn = (mv.xy * (lens_tilt_amplitude * np.sin(2 * phase) / 2)).exp()  # [] Rotor+
        mirror_turn = (mv.xy * ((mirror_tilt + rock_amplitude * np.cos(phase)) / 2)).exp()  # [] Rotor+
        motors = (lens_slide * stops[0] * lens_turn, stops[1], stops[2], stops[3] * mirror_turn)
        planes, legs, composed = optical_train_trace(fan, motors, train_elements)
        outgoing = composed(fan)                                                # [n_fan_rays] Line-
        yield train_subject, planes, legs, composed, outgoing[0] ^ outgoing[-1]


recorded = show_states(
    moving_train_states(n_frames),
    render.train_animation,
    name="thin_lens",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

Rays leave the blue object dot on the left and change colour across each grey element (the sliding and tilting converging lens, the fixed prism and diverging lens, and the rocking diagonal mirror), converging cleanly through the purple image dot in the folded upper beam throughout the motion.

In [ ]:
# checks
# Every refracted ray passes through the image crossing, signed object and image distances satisfy the
# thin-lens equation, two-lens parallel focusing matches Gullstrand's back focal distance on the optical axis,
# and the moving train keeps all outgoing rays concurrent at the image in every recorded frame.
assert ((focus & axis).abs() < 1e-12).all()
assert ((image & refracted).abs() < 1e-11).all()

object_distance = -(first_plane & subject) / (mv.w & subject)                   # [] Scalar+
image_distance = (first_plane & image) / (mv.w & image)                         # [] Scalar+
assert ((1 / object_distance + 1 / image_distance - 1 / first_focal).abs() < 1e-12).all()

gap = (first_plane - second_plane) & origin                                     # [] Scalar+
effective = 1 / (1 / first_focal + 1 / second_focal - gap / (first_focal * second_focal))  # [] Scalar+
back_focal = effective * (first_focal - gap) / first_focal                      # [] Scalar+
assert (((second_plane & focus) / (mv.w & focus) - back_focal).abs() < 1e-12).all()

for _, _, legs, composed, picture in recorded:
    assert ((picture & composed(legs[0])).abs() < 1e-10).all()